In [ ]:
from __future__ import annotations

import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd


try:
    import torch as _torch
    import torch.nn as _nn
    from torch.utils.data import DataLoader as _DataLoader
    from torch.utils.data import TensorDataset as _TensorDataset
except Exception:  # pragma: no cover - optional dependency import guard
    _torch = None
    _nn = None
    _DataLoader = None
    _TensorDataset = None


OUTER_FOLDS = (1, 2, 3, 4, 5)
DL_MODEL_FAMILIES = ["dl_tabnet", "dl_ft_transformer"]
DL_TRIALS_TABNET = 8
DL_TRIALS_FT = 8
TARGET_COL = "Target_Log"
DEVICE = "cpu"
TABNET_DEVICE = "cpu"
TABNET_BASELINE_CFG = {
    "n_d": 16,
    "n_a": 16,
    "n_steps": 4,
    "gamma": 1.5,
    "lr": 1e-3,
    "lambda_sparse": 1e-3,
    "max_epochs": 140,
    "patience": 25,
    "batch_size": 1024,
    "virtual_batch_size": 128,
}
FT_BASELINE_CFG = {
    "d_model": 64,
    "nhead": 8,
    "layers": 2,
    "dropout": 0.1,
    "ff_mult": 4,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 45,
    "batch_size": 1024,
}

batched_predict_log = None
build_prediction_frame = None
compute_regression_metrics = None
ensure_family_result_dir = None
load_full_feature_processed = None
save_model_artifact = None
summarize_fold_metrics = None
RUN_NOTEBOOK_PIPELINE = __name__ == "__main__"

torch = _torch
nn = _nn
DataLoader = _DataLoader
TensorDataset = _TensorDataset
TabNetRegressor = None
optuna = None
MedianPruner = None
TPESampler = None


if _nn is not None:
    class FTTransformerRegressor(_nn.Module):
        def __init__(self, n_features: int, cfg: dict[str, object]):
            super().__init__()
            d_model = int(cfg["d_model"])
            self.feature_weight = _nn.Parameter(_torch.randn(n_features, d_model) * 0.02)
            self.feature_bias = _nn.Parameter(_torch.zeros(n_features, d_model))
            self.cls_token = _nn.Parameter(_torch.zeros(1, 1, d_model))
            encoder_layer = _nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=int(cfg["nhead"]),
                dim_feedforward=d_model * int(cfg["ff_mult"]),
                dropout=float(cfg["dropout"]),
                batch_first=True,
                activation="gelu",
            )
            self.encoder = _nn.TransformerEncoder(encoder_layer, num_layers=int(cfg["layers"]))
            self.head = _nn.Sequential(_nn.LayerNorm(d_model), _nn.Linear(d_model, 1))

        def forward(self, x):
            tokens = x.unsqueeze(-1) * self.feature_weight.unsqueeze(0) + self.feature_bias.unsqueeze(0)
            hidden = _torch.cat([self.cls_token.expand(x.size(0), 1, -1), tokens], dim=1)
            return self.head(self.encoder(hidden)[:, 0, :]).squeeze(-1)
else:
    class FTTransformerRegressor:  # pragma: no cover - unavailable without torch
        pass


In [ ]:
def resolve_current_code_dir() -> Path:
    if "__file__" in globals():
        return Path(__file__).resolve().parent

    code_dir = Path.cwd().resolve()
    if code_dir.name != "1_Code":
        raise RuntimeError(
            "cannot resolve current code directory without __file__; "
            "current working directory must be the Step5 1_Code directory"
        )
    return code_dir


In [ ]:
def initialize_runtime() -> Path:
    global TARGET_COL
    global batched_predict_log
    global build_prediction_frame
    global compute_regression_metrics
    global ensure_family_result_dir
    global load_full_feature_processed
    global save_model_artifact
    global summarize_fold_metrics
    global torch
    global nn
    global DataLoader
    global TensorDataset
    global TabNetRegressor
    global optuna
    global MedianPruner
    global TPESampler
    global DEVICE
    global TABNET_DEVICE

    current_code_dir = resolve_current_code_dir()
    if str(current_code_dir) not in sys.path:
        sys.path.insert(0, str(current_code_dir))

    try:
        import torch as runtime_torch
        import torch.nn as runtime_nn
        from torch.utils.data import DataLoader as runtime_data_loader
        from torch.utils.data import TensorDataset as runtime_tensor_dataset
    except Exception:
        runtime_torch = None
        runtime_nn = None
        runtime_data_loader = None
        runtime_tensor_dataset = None

    try:
        from pytorch_tabnet.tab_model import TabNetRegressor as runtime_tabnet_regressor
    except Exception:
        runtime_tabnet_regressor = None

    try:
        import optuna as runtime_optuna
        from optuna.pruners import MedianPruner as runtime_median_pruner
        from optuna.samplers import TPESampler as runtime_tpe_sampler
    except Exception:
        runtime_optuna = None
        runtime_median_pruner = None
        runtime_tpe_sampler = None

    from _step5_shared import (
        TARGET_COL as shared_target_col,
        batched_predict_log as shared_batched_predict_log,
        build_prediction_frame as shared_build_prediction_frame,
        compute_regression_metrics as shared_compute_regression_metrics,
        ensure_family_result_dir as shared_ensure_family_result_dir,
        load_full_feature_processed as shared_load_full_feature_processed,
        save_model_artifact as shared_save_model_artifact,
        summarize_fold_metrics as shared_summarize_fold_metrics,
    )

    TARGET_COL = shared_target_col
    if batched_predict_log is None:
        batched_predict_log = shared_batched_predict_log
    if build_prediction_frame is None:
        build_prediction_frame = shared_build_prediction_frame
    if compute_regression_metrics is None:
        compute_regression_metrics = shared_compute_regression_metrics
    if ensure_family_result_dir is None:
        ensure_family_result_dir = shared_ensure_family_result_dir
    if load_full_feature_processed is None:
        load_full_feature_processed = shared_load_full_feature_processed
    if save_model_artifact is None:
        save_model_artifact = shared_save_model_artifact
    if summarize_fold_metrics is None:
        summarize_fold_metrics = shared_summarize_fold_metrics

    torch = runtime_torch
    nn = runtime_nn
    DataLoader = runtime_data_loader
    TensorDataset = runtime_tensor_dataset
    TabNetRegressor = runtime_tabnet_regressor
    optuna = runtime_optuna
    MedianPruner = runtime_median_pruner
    TPESampler = runtime_tpe_sampler

    DEVICE = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
    TABNET_DEVICE = "cuda" if DEVICE == "cuda" else "cpu"
    return current_code_dir


In [ ]:
def _require_torch() -> None:
    if any(dep is None for dep in (torch, nn, DataLoader, TensorDataset)):
        raise RuntimeError("torch runtime is unavailable for DL family compare")


In [ ]:
def _require_tabnet() -> None:
    if TabNetRegressor is None:
        raise RuntimeError("pytorch-tabnet is unavailable for dl_tabnet")


In [ ]:
def _require_optuna() -> None:
    if any(dep is None for dep in (optuna, MedianPruner, TPESampler)):
        raise RuntimeError("optuna is unavailable for tuned DL runs")


In [ ]:
def _set_random_seeds(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    if torch is not None:
        torch.manual_seed(int(seed))
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(int(seed))
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False


In [ ]:
def _release_model_resources(model) -> None:
    if model is None:
        return
    if hasattr(model, "to") and torch is not None:
        try:
            model.to("cpu")
        except Exception:
            pass


In [ ]:
def _run_memory_cleanup() -> None:
    import gc

    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
def cleanup_runtime_objects(*object_names: str) -> None:
    for object_name in object_names:
        if object_name in globals():
            globals().pop(object_name, None)
    _run_memory_cleanup()


In [ ]:
def expected_output_files() -> list[str]:
    return [
        "2_dl_model_artifact_manifest.csv",
        "2_dl_predictions_outer_test.csv",
        "2_dl_metrics_by_model_by_outer_fold.csv",
        "2_dl_metrics_by_model_mean_std.csv",
        "2_dl_model_ranking.csv",
        "2_dl_best_models.csv",
    ]


In [ ]:
def prepare_numeric_arrays(
    frame: pd.DataFrame,
    feature_cols: list[str],
    fill_values: pd.Series | None = None,
) -> tuple[np.ndarray, np.ndarray, pd.Series]:
    x_frame = frame.loc[:, feature_cols].apply(pd.to_numeric, errors="coerce")
    if fill_values is None:
        fill_values = x_frame.median(axis=0, skipna=True).fillna(0.0)
    else:
        fill_values = pd.Series(fill_values, index=feature_cols, dtype=float).fillna(0.0)
    x_np = x_frame.fillna(fill_values).to_numpy(dtype=np.float32)
    y_np = pd.to_numeric(frame[TARGET_COL], errors="raise").to_numpy(dtype=np.float32)
    return x_np, y_np, fill_values


In [ ]:
def iter_inner_split_frames(train_df: pd.DataFrame, inner_index: pd.DataFrame) -> list[tuple[pd.DataFrame, pd.DataFrame]]:
    split_frames: list[tuple[pd.DataFrame, pd.DataFrame]] = []
    fold_ids = sorted(pd.to_numeric(inner_index["inner_fold"], errors="raise").astype(int).unique().tolist())

    for inner_fold in fold_ids:
        train_ids = inner_index.loc[
            (pd.to_numeric(inner_index["inner_fold"], errors="raise").astype(int) == int(inner_fold))
            & (inner_index["split"].astype(str) == "train"),
            "row_uid",
        ].astype(str)
        valid_ids = inner_index.loc[
            (pd.to_numeric(inner_index["inner_fold"], errors="raise").astype(int) == int(inner_fold))
            & (inner_index["split"].astype(str) == "valid"),
            "row_uid",
        ].astype(str)
        fold_train = train_df.loc[train_df["row_uid"].astype(str).isin(train_ids)].copy()
        fold_valid = train_df.loc[train_df["row_uid"].astype(str).isin(valid_ids)].copy()
        if fold_train.empty or fold_valid.empty:
            raise RuntimeError(f"inner fold {inner_fold} produced empty train/valid split")
        split_frames.append((fold_train, fold_valid))

    if not split_frames:
        raise RuntimeError("inner fold index produced no train/valid splits")
    return split_frames


In [ ]:
def fit_tabnet_model(
    x_train: np.ndarray,
    y_train: np.ndarray,
    cfg: dict[str, object],
    random_seed: int,
    x_valid: np.ndarray | None = None,
    y_valid: np.ndarray | None = None,
):
    _require_tabnet()
    _set_random_seeds(int(random_seed))
    if len(x_train) <= 0:
        raise RuntimeError("tabnet training requires at least one training row")

    sample_count = int(len(x_train))
    batch_size = min(int(cfg.get("batch_size", sample_count)), sample_count)
    batch_size = max(1, batch_size)
    virtual_batch_size = int(cfg.get("virtual_batch_size", batch_size))
    virtual_batch_size = min(virtual_batch_size, batch_size, sample_count)
    virtual_batch_size = max(1, virtual_batch_size)
    fit_kwargs: dict[str, object] = {
        "max_epochs": int(cfg["max_epochs"]),
        "patience": int(cfg.get("patience", 15)),
        "batch_size": batch_size,
        "virtual_batch_size": virtual_batch_size,
    }

    if x_valid is not None or y_valid is not None:
        if x_valid is None or y_valid is None:
            raise RuntimeError("tabnet validation inputs must include both x_valid and y_valid")
        if len(x_valid) <= 0:
            raise RuntimeError("tabnet validation set must contain at least one row")
        fit_kwargs["eval_set"] = [
            (
                np.asarray(x_valid, dtype=np.float32),
                np.asarray(y_valid, dtype=np.float32).reshape(-1, 1),
            )
        ]
        fit_kwargs["eval_name"] = ["valid"]
        fit_kwargs["eval_metric"] = ["rmse"]

    model = TabNetRegressor(
        n_d=int(cfg["n_d"]),
        n_a=int(cfg["n_a"]),
        n_steps=int(cfg["n_steps"]),
        gamma=float(cfg["gamma"]),
        lambda_sparse=float(cfg.get("lambda_sparse", 1e-3)),
        optimizer_params={"lr": float(cfg["lr"])},
        seed=int(random_seed),
        verbose=0,
        device_name=TABNET_DEVICE,
    )
    model.fit(
        x_train,
        y_train.reshape(-1, 1),
        **fit_kwargs,
    )
    setattr(model, "_step5_model_cfg", dict(cfg))
    setattr(model, "_step5_model_type", "dl_tabnet")
    setattr(model, "_step5_n_features", int(x_train.shape[1]))
    return model


In [ ]:
def fit_ft_transformer_model(x_train: np.ndarray, y_train: np.ndarray, cfg: dict[str, object], random_seed: int):
    _require_torch()
    _set_random_seeds(int(random_seed))

    model = FTTransformerRegressor(x_train.shape[1], cfg).to(DEVICE)
    dataset = TensorDataset(torch.from_numpy(x_train).float(), torch.from_numpy(y_train).float())
    dataloader = DataLoader(dataset, batch_size=int(cfg["batch_size"]), shuffle=True)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg["lr"]),
        weight_decay=float(cfg["weight_decay"]),
    )
    loss_fn = nn.MSELoss()
    model.train()
    for _ in range(int(cfg["epochs"])):
        for xb, yb in dataloader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
    setattr(model, "_step5_model_cfg", dict(cfg))
    setattr(model, "_step5_model_type", "dl_ft_transformer")
    setattr(model, "_step5_n_features", int(x_train.shape[1]))
    return model


In [ ]:
def build_serializable_dl_artifact(model_id: str, model) -> object:
    if model_id == "dl_ft_transformer" and hasattr(model, "state_dict") and hasattr(model, "_step5_n_features"):
        return {
            "model_family": "dl",
            "model_id": str(model_id),
            "artifact_kind": "state_dict_bundle",
            "n_features": int(getattr(model, "_step5_n_features")),
            "model_cfg": dict(getattr(model, "_step5_model_cfg")),
            "state_dict": {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            },
            "fill_values": getattr(model, "_step5_fill_values", None),
        }
    return model


In [ ]:
def restore_ft_transformer_artifact(artifact: object):
    _require_torch()
    if not isinstance(artifact, dict) or artifact.get("artifact_kind") != "state_dict_bundle":
        raise RuntimeError("artifact must be a FT state_dict_bundle")

    model = FTTransformerRegressor(int(artifact["n_features"]), dict(artifact["model_cfg"])).to(DEVICE)
    model.load_state_dict(dict(artifact["state_dict"]))
    setattr(model, "_step5_model_cfg", dict(artifact["model_cfg"]))
    setattr(model, "_step5_model_type", "dl_ft_transformer")
    setattr(model, "_step5_n_features", int(artifact["n_features"]))
    fill_values = artifact.get("fill_values")
    if fill_values is not None:
        setattr(model, "_step5_fill_values", pd.Series(fill_values, dtype=float))
    return model


In [ ]:
def predict_ft_transformer(model, x_other: np.ndarray) -> np.ndarray:
    _require_torch()
    model.eval()
    with torch.no_grad():
        x_tensor = torch.from_numpy(x_other).float().to(DEVICE)
        pred = model(x_tensor).detach().cpu().numpy()
    return np.asarray(pred, dtype=float).reshape(-1)


In [ ]:
def evaluate_tabnet_cfg(
    train_df: pd.DataFrame,
    inner_index: pd.DataFrame,
    feature_cols: list[str],
    cfg: dict[str, object],
    random_seed: int,
) -> float:
    fold_maes: list[float] = []
    for offset, (fold_train, fold_valid) in enumerate(iter_inner_split_frames(train_df, inner_index), start=1):
        x_train, y_train, fill_values = prepare_numeric_arrays(fold_train, feature_cols)
        x_valid, y_valid, _ = prepare_numeric_arrays(fold_valid, feature_cols, fill_values=fill_values)
        model = fit_tabnet_model(
            x_train,
            y_train,
            cfg,
            random_seed=random_seed + offset,
            x_valid=x_valid,
            y_valid=y_valid,
        )
        try:
            pred = batched_predict_log(model, x_valid, batch_size=256)
            fold_maes.append(float(np.mean(np.abs(y_valid - pred))))
        finally:
            _release_model_resources(model)
            _run_memory_cleanup()
    return float(np.mean(fold_maes))


In [ ]:
def evaluate_ft_cfg(
    train_df: pd.DataFrame,
    inner_index: pd.DataFrame,
    feature_cols: list[str],
    cfg: dict[str, object],
    random_seed: int,
) -> float:
    fold_maes: list[float] = []
    for offset, (fold_train, fold_valid) in enumerate(iter_inner_split_frames(train_df, inner_index), start=1):
        x_train, y_train, fill_values = prepare_numeric_arrays(fold_train, feature_cols)
        x_valid, y_valid, _ = prepare_numeric_arrays(fold_valid, feature_cols, fill_values=fill_values)
        model = fit_ft_transformer_model(x_train, y_train, cfg, random_seed=random_seed + offset)
        try:
            pred = predict_ft_transformer(model, x_valid)
            fold_maes.append(float(np.mean(np.abs(y_valid - pred))))
        finally:
            _release_model_resources(model)
            _run_memory_cleanup()
    return float(np.mean(fold_maes))


In [ ]:
def fit_tabnet_baseline(
    train_df: pd.DataFrame,
    feature_cols: list[str],
    random_seed: int,
    valid_df: pd.DataFrame | None = None,
):
    x_train, y_train, fill_values = prepare_numeric_arrays(train_df, feature_cols)
    model = fit_tabnet_model(
        x_train,
        y_train,
        TABNET_BASELINE_CFG,
        random_seed=random_seed,
    )
    setattr(model, "_step5_fill_values", fill_values)
    return model


In [ ]:
def fit_ft_transformer_baseline(train_df: pd.DataFrame, feature_cols: list[str], random_seed: int):
    x_train, y_train, fill_values = prepare_numeric_arrays(train_df, feature_cols)
    model = fit_ft_transformer_model(x_train, y_train, FT_BASELINE_CFG, random_seed=random_seed)
    setattr(model, "_step5_fill_values", fill_values)
    return model


In [ ]:
def _fit_tabnet_baseline_runtime(
    train_df: pd.DataFrame,
    feature_cols: list[str],
    random_seed: int,
    valid_df: pd.DataFrame | None = None,
):
    baseline_callable = fit_tabnet_baseline
    if baseline_callable is _fit_tabnet_baseline_runtime:
        raise RuntimeError("baseline runtime wrapper recursion detected")

    try:
        return baseline_callable(
            train_df,
            feature_cols,
            random_seed=random_seed,
            valid_df=valid_df,
        )
    except TypeError as exc:
        if "valid_df" not in str(exc):
            raise
        return baseline_callable(
            train_df,
            feature_cols,
            random_seed=random_seed,
        )


In [ ]:
def fit_tabnet_tuned(
    train_df: pd.DataFrame,
    inner_index: pd.DataFrame,
    feature_cols: list[str],
    n_trials: int,
    random_seed: int,
):
    _require_optuna()

    def objective(trial) -> float:
        cfg = {
            "n_d": trial.suggest_categorical("n_d", [8, 16, 24, 32]),
            "n_steps": trial.suggest_int("n_steps", 3, 8),
            "gamma": trial.suggest_float("gamma", 1.1, 2.0),
            "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
            "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-2, log=True),
            "max_epochs": trial.suggest_int("max_epochs", 100, 220),
            "patience": trial.suggest_int("patience", 15, 40),
            "batch_size": trial.suggest_categorical("batch_size", [512, 1024, 2048]),
            "virtual_batch_size": trial.suggest_categorical("virtual_batch_size", [64, 128, 256]),
        }
        cfg["n_a"] = cfg["n_d"]
        return evaluate_tabnet_cfg(train_df, inner_index, feature_cols, cfg, random_seed=random_seed)

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=int(random_seed)),
        pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=int(n_trials), show_progress_bar=False)
    best_cfg = dict(study.best_trial.params)
    best_cfg["n_a"] = best_cfg["n_d"]
    final_x_train, final_y_train, fill_values = prepare_numeric_arrays(train_df, feature_cols)
    model = fit_tabnet_model(
        final_x_train,
        final_y_train,
        best_cfg,
        random_seed=random_seed,
    )
    setattr(model, "_step5_fill_values", fill_values)
    return model


In [ ]:
def fit_ft_transformer_tuned(
    train_df: pd.DataFrame,
    inner_index: pd.DataFrame,
    feature_cols: list[str],
    n_trials: int,
    random_seed: int,
):
    _require_optuna()

    def objective(trial) -> float:
        cfg = {
            "d_model": trial.suggest_categorical("d_model", [32, 64, 96, 128]),
            "nhead": trial.suggest_categorical("nhead", [4, 8]),
            "layers": trial.suggest_int("layers", 1, 4),
            "dropout": trial.suggest_float("dropout", 0.0, 0.35),
            "ff_mult": trial.suggest_categorical("ff_mult", [2, 4, 6]),
            "lr": trial.suggest_float("lr", 2e-4, 3e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 5e-3, log=True),
            "epochs": trial.suggest_int("epochs", 40, 140),
            "batch_size": trial.suggest_categorical("batch_size", [512, 1024, 2048]),
        }
        if int(cfg["d_model"]) % int(cfg["nhead"]) != 0:
            cfg["d_model"] = 64
            cfg["nhead"] = 8
        return evaluate_ft_cfg(train_df, inner_index, feature_cols, cfg, random_seed=random_seed)

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=int(random_seed)),
        pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=int(n_trials), show_progress_bar=False)
    best_cfg = dict(study.best_trial.params)
    if int(best_cfg["d_model"]) % int(best_cfg["nhead"]) != 0:
        best_cfg["d_model"] = 64
        best_cfg["nhead"] = 8
    x_train, y_train, fill_values = prepare_numeric_arrays(train_df, feature_cols)
    model = fit_ft_transformer_model(x_train, y_train, best_cfg, random_seed=random_seed)
    setattr(model, "_step5_fill_values", fill_values)
    return model


In [ ]:
def model_predict_log(model_id: str, model, test_df: pd.DataFrame, feature_cols: list[str]) -> np.ndarray:
    fill_values = getattr(model, "_step5_fill_values", None)
    x_test, _, _ = prepare_numeric_arrays(test_df, feature_cols, fill_values=fill_values)
    if model_id == "dl_ft_transformer":
        return predict_ft_transformer(model, x_test)
    return batched_predict_log(model, x_test, batch_size=256)


In [ ]:
def clip_tabnet_prediction_log(prediction_log: np.ndarray, train_df: pd.DataFrame) -> np.ndarray:
    if TARGET_COL not in train_df.columns:
        raise KeyError(f"train_df is missing required target column: {TARGET_COL}")
    train_target = pd.to_numeric(train_df[TARGET_COL], errors="raise").to_numpy(dtype=float)
    lo = float(np.nanmin(train_target))
    hi = float(np.nanmax(train_target))
    prediction_log = np.asarray(prediction_log, dtype=float).reshape(-1)
    return np.clip(prediction_log, lo, hi)


In [ ]:
def load_dl_fold_bundle(step4_root: Path, outer_fold: int) -> dict[str, object]:
    if any(
        dep is None
        for dep in (
            batched_predict_log,
            build_prediction_frame,
            compute_regression_metrics,
            ensure_family_result_dir,
            load_full_feature_processed,
            save_model_artifact,
            summarize_fold_metrics,
        )
    ):
        initialize_runtime()

    train_df, test_df, inner_index, feature_cols = load_full_feature_processed(
        Path(step4_root),
        outer_fold=int(outer_fold),
    )
    return {
        "outer_fold": int(outer_fold),
        "train_df": train_df,
        "test_df": test_df,
        "inner_index": inner_index,
        "feature_cols": list(feature_cols),
    }


In [ ]:
def run_dl_model_step(
    fold_bundle: dict[str, object],
    model_id: str,
    run_role: str,
    model,
    artifact_dir: Path,
) -> dict[str, object]:
    prediction_log = model_predict_log(model_id, model, pd.DataFrame(fold_bundle["test_df"]), list(fold_bundle["feature_cols"]))
    if model_id == "dl_tabnet":
        prediction_log = clip_tabnet_prediction_log(prediction_log, pd.DataFrame(fold_bundle["train_df"]))
    prediction_frame = build_prediction_frame(
        test_df=pd.DataFrame(fold_bundle["test_df"]),
        prediction_log=prediction_log,
        outer_fold=int(fold_bundle["outer_fold"]),
        model_family="dl",
        model_id=model_id,
        run_role=run_role,
    )
    artifact_path = artifact_dir / f"fold_{int(fold_bundle['outer_fold'])}" / f"{model_id}_{run_role}.joblib"
    save_model_artifact(build_serializable_dl_artifact(model_id, model), artifact_path)
    if not artifact_path.is_file():
        raise RuntimeError(f"artifact path does not exist after save: {artifact_path}")
    metric_row = {
        "outer_fold": int(fold_bundle["outer_fold"]),
        "model_family": "dl",
        "model_id": model_id,
        "run_role": run_role,
        **compute_regression_metrics(prediction_frame),
    }
    artifact_row = {
        "outer_fold": int(fold_bundle["outer_fold"]),
        "model_family": "dl",
        "model_id": model_id,
        "run_role": run_role,
        "artifact_path": str(artifact_path),
    }
    return {
        "prediction_frame": prediction_frame,
        "metric_row": metric_row,
        "artifact_row": artifact_row,
    }


In [ ]:
def append_dl_run_outputs(
    prediction_frames: list[pd.DataFrame],
    metric_rows: list[dict[str, object]],
    artifact_rows: list[dict[str, object]],
    result: dict[str, object],
) -> None:
    prediction_frames.append(result["prediction_frame"])
    metric_rows.append(result["metric_row"])
    artifact_rows.append(result["artifact_row"])


In [ ]:
def finalize_dl_run(model, result: dict[str, object] | None = None) -> None:
    del result
    _release_model_resources(model)
    _run_memory_cleanup()


In [ ]:
def write_dl_family_outputs(
    result_dir: Path,
    prediction_frames: list[pd.DataFrame],
    metric_rows: list[dict[str, object]],
    artifact_rows: list[dict[str, object]],
) -> None:
    if not metric_rows or not prediction_frames or not artifact_rows:
        raise RuntimeError("DL family compare produced no outputs")

    metrics_df = pd.DataFrame(metric_rows)
    summary_df = summarize_fold_metrics(metrics_df)
    ranking_df = summary_df.copy()
    ranking_df.insert(0, "rank", range(1, len(ranking_df) + 1))
    best_models_df = ranking_df.head(2).copy()

    pd.DataFrame(artifact_rows).to_csv(result_dir / "2_dl_model_artifact_manifest.csv", index=False)
    pd.concat(prediction_frames, ignore_index=True).to_csv(
        result_dir / "2_dl_predictions_outer_test.csv",
        index=False,
    )
    metrics_df.to_csv(result_dir / "2_dl_metrics_by_model_by_outer_fold.csv", index=False)
    summary_df.to_csv(result_dir / "2_dl_metrics_by_model_mean_std.csv", index=False)
    ranking_df.to_csv(result_dir / "2_dl_model_ranking.csv", index=False)
    best_models_df.to_csv(result_dir / "2_dl_best_models.csv", index=False)


In [ ]:
def _assert_runtime_outputs_written(result_dir: Path) -> None:
    expected_paths = [result_dir / file_name for file_name in expected_output_files()]
    missing_paths = [str(path) for path in expected_paths if not path.is_file()]
    if missing_paths:
        raise RuntimeError(f"missing DL runtime output files: {missing_paths}")

    for csv_path in expected_paths:
        frame = pd.read_csv(csv_path)
        if frame.empty:
            raise RuntimeError(f"runtime output must not be empty: {csv_path}")


In [ ]:
def run_dl_family(step4_root: Path, step5_root: Path) -> None:
    if any(
        dep is None
        for dep in (
            batched_predict_log,
            build_prediction_frame,
            compute_regression_metrics,
            ensure_family_result_dir,
            load_full_feature_processed,
            save_model_artifact,
            summarize_fold_metrics,
        )
    ):
        initialize_runtime()

    result_dir = ensure_family_result_dir(Path(step5_root), "dl")
    artifact_dir = result_dir / "artifacts"
    artifact_dir.mkdir(parents=True, exist_ok=True)

    prediction_frames: list[pd.DataFrame] = []
    metric_rows: list[dict[str, object]] = []
    artifact_rows: list[dict[str, object]] = []

    for outer_fold in OUTER_FOLDS:
        fold_bundle = load_dl_fold_bundle(step4_root=step4_root, outer_fold=int(outer_fold))
        family_run_specs = [
            (
                "dl_tabnet",
                "baseline",
                lambda: _fit_tabnet_baseline_runtime(
                    pd.DataFrame(fold_bundle["train_df"]),
                    list(fold_bundle["feature_cols"]),
                    random_seed=2100 + int(outer_fold),
                ),
            ),
            (
                "dl_tabnet",
                "tuned",
                lambda: fit_tabnet_tuned(
                    pd.DataFrame(fold_bundle["train_df"]),
                    pd.DataFrame(fold_bundle["inner_index"]),
                    list(fold_bundle["feature_cols"]),
                    n_trials=int(DL_TRIALS_TABNET),
                    random_seed=2200 + int(outer_fold),
                ),
            ),
            (
                "dl_ft_transformer",
                "baseline",
                lambda: fit_ft_transformer_baseline(
                    pd.DataFrame(fold_bundle["train_df"]),
                    list(fold_bundle["feature_cols"]),
                    random_seed=2300 + int(outer_fold),
                ),
            ),
            (
                "dl_ft_transformer",
                "tuned",
                lambda: fit_ft_transformer_tuned(
                    pd.DataFrame(fold_bundle["train_df"]),
                    pd.DataFrame(fold_bundle["inner_index"]),
                    list(fold_bundle["feature_cols"]),
                    n_trials=int(DL_TRIALS_FT),
                    random_seed=2400 + int(outer_fold),
                ),
            ),
        ]

        for model_id, run_role, fit_callable in family_run_specs:
            model = fit_callable()
            result = None
            try:
                result = run_dl_model_step(fold_bundle, model_id, run_role, model, artifact_dir)
                append_dl_run_outputs(prediction_frames, metric_rows, artifact_rows, result)
            finally:
                finalize_dl_run(model, result)

        del fold_bundle
        _run_memory_cleanup()

    write_dl_family_outputs(result_dir, prediction_frames, metric_rows, artifact_rows)


In [ ]:
# --- Step 1: Resolve release-local Step4 inputs and Step5 outputs ---
current_code_dir = initialize_runtime()
step5_root = current_code_dir.parent
release_root = step5_root.parent
step4_root = release_root / "Step4_ModelInputs" / "2_Results"
dl_result_dir = ensure_family_result_dir(step5_root, "dl")
dl_artifact_dir = dl_result_dir / "artifacts"
dl_artifact_dir.mkdir(parents=True, exist_ok=True)
dl_prediction_frames: list[pd.DataFrame] = []
dl_metric_rows: list[dict[str, object]] = []
dl_artifact_rows: list[dict[str, object]] = []
print({"step": "dl_runtime_ready", "step4_root": str(step4_root), "result_dir": str(dl_result_dir)})


In [ ]:
if RUN_NOTEBOOK_PIPELINE:
    # --- Step 2: Load fold 1 inputs ---
    dl_fold_1_bundle = load_dl_fold_bundle(step4_root=step4_root, outer_fold=1)
    print({"step": "dl_fold_loaded", "outer_fold": 1, "train_rows": len(dl_fold_1_bundle["train_df"]), "test_rows": len(dl_fold_1_bundle["test_df"])})


# --- Step 2.1: Build fold 1 inner validation anchor for TabNet baseline ---
    dl_fold_1_baseline_split_frames = iter_inner_split_frames(
        pd.DataFrame(dl_fold_1_bundle["train_df"]),
        pd.DataFrame(dl_fold_1_bundle["inner_index"]),
    )
    dl_fold_1_baseline_train_df, dl_fold_1_baseline_valid_df = dl_fold_1_baseline_split_frames[-1]
    print({"step": "dl_fold_baseline_validation_ready", "outer_fold": 1, "baseline_train_rows": len(dl_fold_1_baseline_train_df), "baseline_valid_rows": len(dl_fold_1_baseline_valid_df)})


In [ ]:
# --- Step 3: Fold 1 TabNet baseline ---
dl_fold_1_tabnet_baseline_model = _fit_tabnet_baseline_runtime(
    dl_fold_1_baseline_train_df,
    list(dl_fold_1_bundle["feature_cols"]),
    random_seed=2101,
    valid_df=dl_fold_1_baseline_valid_df,
)
dl_fold_1_tabnet_baseline = run_dl_model_step(
    dl_fold_1_bundle,
    "dl_tabnet",
    "baseline",
    dl_fold_1_tabnet_baseline_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_1_tabnet_baseline)
print(dl_fold_1_tabnet_baseline["metric_row"])
finalize_dl_run(dl_fold_1_tabnet_baseline_model, dl_fold_1_tabnet_baseline)
cleanup_runtime_objects("dl_fold_1_tabnet_baseline_model", "dl_fold_1_tabnet_baseline")


In [ ]:
# --- Step 4: Fold 1 TabNet tuned ---
dl_fold_1_tabnet_tuned_model = fit_tabnet_tuned(
    pd.DataFrame(dl_fold_1_bundle["train_df"]),
    pd.DataFrame(dl_fold_1_bundle["inner_index"]),
    list(dl_fold_1_bundle["feature_cols"]),
    n_trials=int(DL_TRIALS_TABNET),
    random_seed=2201,
)
dl_fold_1_tabnet_tuned = run_dl_model_step(
    dl_fold_1_bundle,
    "dl_tabnet",
    "tuned",
    dl_fold_1_tabnet_tuned_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_1_tabnet_tuned)
print(dl_fold_1_tabnet_tuned["metric_row"])
finalize_dl_run(dl_fold_1_tabnet_tuned_model, dl_fold_1_tabnet_tuned)
cleanup_runtime_objects("dl_fold_1_tabnet_tuned_model", "dl_fold_1_tabnet_tuned")


In [ ]:
# --- Step 5: Fold 1 FT-Transformer baseline ---
dl_fold_1_ft_transformer_baseline_model = fit_ft_transformer_baseline(
    pd.DataFrame(dl_fold_1_bundle["train_df"]),
    list(dl_fold_1_bundle["feature_cols"]),
    random_seed=2301,
)
dl_fold_1_ft_transformer_baseline = run_dl_model_step(
    dl_fold_1_bundle,
    "dl_ft_transformer",
    "baseline",
    dl_fold_1_ft_transformer_baseline_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_1_ft_transformer_baseline)
print(dl_fold_1_ft_transformer_baseline["metric_row"])
finalize_dl_run(dl_fold_1_ft_transformer_baseline_model, dl_fold_1_ft_transformer_baseline)
cleanup_runtime_objects("dl_fold_1_ft_transformer_baseline_model", "dl_fold_1_ft_transformer_baseline")


In [ ]:
# --- Step 6: Fold 1 FT-Transformer tuned ---
dl_fold_1_ft_transformer_tuned_model = fit_ft_transformer_tuned(
    pd.DataFrame(dl_fold_1_bundle["train_df"]),
    pd.DataFrame(dl_fold_1_bundle["inner_index"]),
    list(dl_fold_1_bundle["feature_cols"]),
    n_trials=int(DL_TRIALS_FT),
    random_seed=2401,
)
dl_fold_1_ft_transformer_tuned = run_dl_model_step(
    dl_fold_1_bundle,
    "dl_ft_transformer",
    "tuned",
    dl_fold_1_ft_transformer_tuned_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_1_ft_transformer_tuned)
print(dl_fold_1_ft_transformer_tuned["metric_row"])
finalize_dl_run(dl_fold_1_ft_transformer_tuned_model, dl_fold_1_ft_transformer_tuned)
cleanup_runtime_objects("dl_fold_1_ft_transformer_tuned_model", "dl_fold_1_ft_transformer_tuned")
cleanup_runtime_objects("dl_fold_1_bundle", "dl_fold_1_baseline_split_frames", "dl_fold_1_baseline_train_df", "dl_fold_1_baseline_valid_df")


In [ ]:
if RUN_NOTEBOOK_PIPELINE:
    # --- Step 7: Load fold 2 inputs ---
    dl_fold_2_bundle = load_dl_fold_bundle(step4_root=step4_root, outer_fold=2)
    print({"step": "dl_fold_loaded", "outer_fold": 2, "train_rows": len(dl_fold_2_bundle["train_df"]), "test_rows": len(dl_fold_2_bundle["test_df"])})


# --- Step 7.1: Build fold 2 inner validation anchor ---
    dl_fold_2_baseline_split_frames = iter_inner_split_frames(
        pd.DataFrame(dl_fold_2_bundle["train_df"]),
        pd.DataFrame(dl_fold_2_bundle["inner_index"]),
    )
    dl_fold_2_baseline_train_df, dl_fold_2_baseline_valid_df = dl_fold_2_baseline_split_frames[-1]
    print({"step": "dl_fold_baseline_validation_ready", "outer_fold": 2, "baseline_train_rows": len(dl_fold_2_baseline_train_df), "baseline_valid_rows": len(dl_fold_2_baseline_valid_df)})


In [ ]:
# --- Step 8: Fold 2 TabNet baseline ---
dl_fold_2_tabnet_baseline_model = _fit_tabnet_baseline_runtime(
    dl_fold_2_baseline_train_df,
    list(dl_fold_2_bundle["feature_cols"]),
    random_seed=2102,
    valid_df=dl_fold_2_baseline_valid_df,
)
dl_fold_2_tabnet_baseline = run_dl_model_step(
    dl_fold_2_bundle,
    "dl_tabnet",
    "baseline",
    dl_fold_2_tabnet_baseline_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_2_tabnet_baseline)
print(dl_fold_2_tabnet_baseline["metric_row"])
finalize_dl_run(dl_fold_2_tabnet_baseline_model, dl_fold_2_tabnet_baseline)
cleanup_runtime_objects("dl_fold_2_tabnet_baseline_model", "dl_fold_2_tabnet_baseline")


In [ ]:
# --- Step 9: Fold 2 TabNet tuned ---
dl_fold_2_tabnet_tuned_model = fit_tabnet_tuned(
    pd.DataFrame(dl_fold_2_bundle["train_df"]),
    pd.DataFrame(dl_fold_2_bundle["inner_index"]),
    list(dl_fold_2_bundle["feature_cols"]),
    n_trials=int(DL_TRIALS_TABNET),
    random_seed=2202,
)
dl_fold_2_tabnet_tuned = run_dl_model_step(
    dl_fold_2_bundle,
    "dl_tabnet",
    "tuned",
    dl_fold_2_tabnet_tuned_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_2_tabnet_tuned)
print(dl_fold_2_tabnet_tuned["metric_row"])
finalize_dl_run(dl_fold_2_tabnet_tuned_model, dl_fold_2_tabnet_tuned)
cleanup_runtime_objects("dl_fold_2_tabnet_tuned_model", "dl_fold_2_tabnet_tuned")


In [ ]:
# --- Step 10: Fold 2 FT-Transformer baseline ---
dl_fold_2_ft_transformer_baseline_model = fit_ft_transformer_baseline(
    pd.DataFrame(dl_fold_2_bundle["train_df"]),
    list(dl_fold_2_bundle["feature_cols"]),
    random_seed=2302,
)
dl_fold_2_ft_transformer_baseline = run_dl_model_step(
    dl_fold_2_bundle,
    "dl_ft_transformer",
    "baseline",
    dl_fold_2_ft_transformer_baseline_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_2_ft_transformer_baseline)
print(dl_fold_2_ft_transformer_baseline["metric_row"])
finalize_dl_run(dl_fold_2_ft_transformer_baseline_model, dl_fold_2_ft_transformer_baseline)
cleanup_runtime_objects("dl_fold_2_ft_transformer_baseline_model", "dl_fold_2_ft_transformer_baseline")


In [ ]:
# --- Step 11: Fold 2 FT-Transformer tuned ---
dl_fold_2_ft_transformer_tuned_model = fit_ft_transformer_tuned(
    pd.DataFrame(dl_fold_2_bundle["train_df"]),
    pd.DataFrame(dl_fold_2_bundle["inner_index"]),
    list(dl_fold_2_bundle["feature_cols"]),
    n_trials=int(DL_TRIALS_FT),
    random_seed=2402,
)
dl_fold_2_ft_transformer_tuned = run_dl_model_step(
    dl_fold_2_bundle,
    "dl_ft_transformer",
    "tuned",
    dl_fold_2_ft_transformer_tuned_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_2_ft_transformer_tuned)
print(dl_fold_2_ft_transformer_tuned["metric_row"])
finalize_dl_run(dl_fold_2_ft_transformer_tuned_model, dl_fold_2_ft_transformer_tuned)
cleanup_runtime_objects("dl_fold_2_ft_transformer_tuned_model", "dl_fold_2_ft_transformer_tuned")
cleanup_runtime_objects("dl_fold_2_bundle", "dl_fold_2_baseline_split_frames", "dl_fold_2_baseline_train_df", "dl_fold_2_baseline_valid_df")
print({"step": "dl_fold_completed", "outer_fold": 2})


In [ ]:
if RUN_NOTEBOOK_PIPELINE:
    # --- Step 12: Load fold 3 inputs ---
    dl_fold_3_bundle = load_dl_fold_bundle(step4_root=step4_root, outer_fold=3)
    print({"step": "dl_fold_loaded", "outer_fold": 3, "train_rows": len(dl_fold_3_bundle["train_df"]), "test_rows": len(dl_fold_3_bundle["test_df"])})


# --- Step 12.1: Build fold 3 inner validation anchor ---
    dl_fold_3_baseline_split_frames = iter_inner_split_frames(
        pd.DataFrame(dl_fold_3_bundle["train_df"]),
        pd.DataFrame(dl_fold_3_bundle["inner_index"]),
    )
    dl_fold_3_baseline_train_df, dl_fold_3_baseline_valid_df = dl_fold_3_baseline_split_frames[-1]
    print({"step": "dl_fold_baseline_validation_ready", "outer_fold": 3, "baseline_train_rows": len(dl_fold_3_baseline_train_df), "baseline_valid_rows": len(dl_fold_3_baseline_valid_df)})


In [ ]:
# --- Step 13: Fold 3 TabNet baseline ---
dl_fold_3_tabnet_baseline_model = _fit_tabnet_baseline_runtime(
    dl_fold_3_baseline_train_df,
    list(dl_fold_3_bundle["feature_cols"]),
    random_seed=2103,
    valid_df=dl_fold_3_baseline_valid_df,
)
dl_fold_3_tabnet_baseline = run_dl_model_step(
    dl_fold_3_bundle,
    "dl_tabnet",
    "baseline",
    dl_fold_3_tabnet_baseline_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_3_tabnet_baseline)
print(dl_fold_3_tabnet_baseline["metric_row"])
finalize_dl_run(dl_fold_3_tabnet_baseline_model, dl_fold_3_tabnet_baseline)
cleanup_runtime_objects("dl_fold_3_tabnet_baseline_model", "dl_fold_3_tabnet_baseline")


In [ ]:
# --- Step 14: Fold 3 TabNet tuned ---
dl_fold_3_tabnet_tuned_model = fit_tabnet_tuned(
    pd.DataFrame(dl_fold_3_bundle["train_df"]),
    pd.DataFrame(dl_fold_3_bundle["inner_index"]),
    list(dl_fold_3_bundle["feature_cols"]),
    n_trials=int(DL_TRIALS_TABNET),
    random_seed=2203,
)
dl_fold_3_tabnet_tuned = run_dl_model_step(
    dl_fold_3_bundle,
    "dl_tabnet",
    "tuned",
    dl_fold_3_tabnet_tuned_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_3_tabnet_tuned)
print(dl_fold_3_tabnet_tuned["metric_row"])
finalize_dl_run(dl_fold_3_tabnet_tuned_model, dl_fold_3_tabnet_tuned)
cleanup_runtime_objects("dl_fold_3_tabnet_tuned_model", "dl_fold_3_tabnet_tuned")


In [ ]:
# --- Step 15: Fold 3 FT-Transformer baseline ---
dl_fold_3_ft_transformer_baseline_model = fit_ft_transformer_baseline(
    pd.DataFrame(dl_fold_3_bundle["train_df"]),
    list(dl_fold_3_bundle["feature_cols"]),
    random_seed=2303,
)
dl_fold_3_ft_transformer_baseline = run_dl_model_step(
    dl_fold_3_bundle,
    "dl_ft_transformer",
    "baseline",
    dl_fold_3_ft_transformer_baseline_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_3_ft_transformer_baseline)
print(dl_fold_3_ft_transformer_baseline["metric_row"])
finalize_dl_run(dl_fold_3_ft_transformer_baseline_model, dl_fold_3_ft_transformer_baseline)
cleanup_runtime_objects("dl_fold_3_ft_transformer_baseline_model", "dl_fold_3_ft_transformer_baseline")


In [ ]:
# --- Step 16: Fold 3 FT-Transformer tuned ---
dl_fold_3_ft_transformer_tuned_model = fit_ft_transformer_tuned(
    pd.DataFrame(dl_fold_3_bundle["train_df"]),
    pd.DataFrame(dl_fold_3_bundle["inner_index"]),
    list(dl_fold_3_bundle["feature_cols"]),
    n_trials=int(DL_TRIALS_FT),
    random_seed=2403,
)
dl_fold_3_ft_transformer_tuned = run_dl_model_step(
    dl_fold_3_bundle,
    "dl_ft_transformer",
    "tuned",
    dl_fold_3_ft_transformer_tuned_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_3_ft_transformer_tuned)
print(dl_fold_3_ft_transformer_tuned["metric_row"])
finalize_dl_run(dl_fold_3_ft_transformer_tuned_model, dl_fold_3_ft_transformer_tuned)
cleanup_runtime_objects("dl_fold_3_ft_transformer_tuned_model", "dl_fold_3_ft_transformer_tuned")
cleanup_runtime_objects("dl_fold_3_bundle", "dl_fold_3_baseline_split_frames", "dl_fold_3_baseline_train_df", "dl_fold_3_baseline_valid_df")
print({"step": "dl_fold_completed", "outer_fold": 3})


In [ ]:
if RUN_NOTEBOOK_PIPELINE:
    # --- Step 17: Load fold 4 inputs ---
    dl_fold_4_bundle = load_dl_fold_bundle(step4_root=step4_root, outer_fold=4)
    print({"step": "dl_fold_loaded", "outer_fold": 4, "train_rows": len(dl_fold_4_bundle["train_df"]), "test_rows": len(dl_fold_4_bundle["test_df"])})


# --- Step 17.1: Build fold 4 inner validation anchor ---
    dl_fold_4_baseline_split_frames = iter_inner_split_frames(
        pd.DataFrame(dl_fold_4_bundle["train_df"]),
        pd.DataFrame(dl_fold_4_bundle["inner_index"]),
    )
    dl_fold_4_baseline_train_df, dl_fold_4_baseline_valid_df = dl_fold_4_baseline_split_frames[-1]
    print({"step": "dl_fold_baseline_validation_ready", "outer_fold": 4, "baseline_train_rows": len(dl_fold_4_baseline_train_df), "baseline_valid_rows": len(dl_fold_4_baseline_valid_df)})


In [ ]:
# --- Step 18: Fold 4 TabNet baseline ---
dl_fold_4_tabnet_baseline_model = _fit_tabnet_baseline_runtime(
    dl_fold_4_baseline_train_df,
    list(dl_fold_4_bundle["feature_cols"]),
    random_seed=2104,
    valid_df=dl_fold_4_baseline_valid_df,
)
dl_fold_4_tabnet_baseline = run_dl_model_step(
    dl_fold_4_bundle,
    "dl_tabnet",
    "baseline",
    dl_fold_4_tabnet_baseline_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_4_tabnet_baseline)
print(dl_fold_4_tabnet_baseline["metric_row"])
finalize_dl_run(dl_fold_4_tabnet_baseline_model, dl_fold_4_tabnet_baseline)
cleanup_runtime_objects("dl_fold_4_tabnet_baseline_model", "dl_fold_4_tabnet_baseline")


In [ ]:
# --- Step 19: Fold 4 TabNet tuned ---
dl_fold_4_tabnet_tuned_model = fit_tabnet_tuned(
    pd.DataFrame(dl_fold_4_bundle["train_df"]),
    pd.DataFrame(dl_fold_4_bundle["inner_index"]),
    list(dl_fold_4_bundle["feature_cols"]),
    n_trials=int(DL_TRIALS_TABNET),
    random_seed=2204,
)
dl_fold_4_tabnet_tuned = run_dl_model_step(
    dl_fold_4_bundle,
    "dl_tabnet",
    "tuned",
    dl_fold_4_tabnet_tuned_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_4_tabnet_tuned)
print(dl_fold_4_tabnet_tuned["metric_row"])
finalize_dl_run(dl_fold_4_tabnet_tuned_model, dl_fold_4_tabnet_tuned)
cleanup_runtime_objects("dl_fold_4_tabnet_tuned_model", "dl_fold_4_tabnet_tuned")


In [ ]:
# --- Step 20: Fold 4 FT-Transformer baseline ---
dl_fold_4_ft_transformer_baseline_model = fit_ft_transformer_baseline(
    pd.DataFrame(dl_fold_4_bundle["train_df"]),
    list(dl_fold_4_bundle["feature_cols"]),
    random_seed=2304,
)
dl_fold_4_ft_transformer_baseline = run_dl_model_step(
    dl_fold_4_bundle,
    "dl_ft_transformer",
    "baseline",
    dl_fold_4_ft_transformer_baseline_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_4_ft_transformer_baseline)
print(dl_fold_4_ft_transformer_baseline["metric_row"])
finalize_dl_run(dl_fold_4_ft_transformer_baseline_model, dl_fold_4_ft_transformer_baseline)
cleanup_runtime_objects("dl_fold_4_ft_transformer_baseline_model", "dl_fold_4_ft_transformer_baseline")


In [ ]:
# --- Step 21: Fold 4 FT-Transformer tuned ---
dl_fold_4_ft_transformer_tuned_model = fit_ft_transformer_tuned(
    pd.DataFrame(dl_fold_4_bundle["train_df"]),
    pd.DataFrame(dl_fold_4_bundle["inner_index"]),
    list(dl_fold_4_bundle["feature_cols"]),
    n_trials=int(DL_TRIALS_FT),
    random_seed=2404,
)
dl_fold_4_ft_transformer_tuned = run_dl_model_step(
    dl_fold_4_bundle,
    "dl_ft_transformer",
    "tuned",
    dl_fold_4_ft_transformer_tuned_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_4_ft_transformer_tuned)
print(dl_fold_4_ft_transformer_tuned["metric_row"])
finalize_dl_run(dl_fold_4_ft_transformer_tuned_model, dl_fold_4_ft_transformer_tuned)
cleanup_runtime_objects("dl_fold_4_ft_transformer_tuned_model", "dl_fold_4_ft_transformer_tuned")
cleanup_runtime_objects("dl_fold_4_bundle", "dl_fold_4_baseline_split_frames", "dl_fold_4_baseline_train_df", "dl_fold_4_baseline_valid_df")
print({"step": "dl_fold_completed", "outer_fold": 4})


In [ ]:
if RUN_NOTEBOOK_PIPELINE:
    # --- Step 22: Load fold 5 inputs ---
    dl_fold_5_bundle = load_dl_fold_bundle(step4_root=step4_root, outer_fold=5)
    print({"step": "dl_fold_loaded", "outer_fold": 5, "train_rows": len(dl_fold_5_bundle["train_df"]), "test_rows": len(dl_fold_5_bundle["test_df"])})


# --- Step 22.1: Build fold 5 inner validation anchor ---
    dl_fold_5_baseline_split_frames = iter_inner_split_frames(
        pd.DataFrame(dl_fold_5_bundle["train_df"]),
        pd.DataFrame(dl_fold_5_bundle["inner_index"]),
    )
    dl_fold_5_baseline_train_df, dl_fold_5_baseline_valid_df = dl_fold_5_baseline_split_frames[-1]
    print({"step": "dl_fold_baseline_validation_ready", "outer_fold": 5, "baseline_train_rows": len(dl_fold_5_baseline_train_df), "baseline_valid_rows": len(dl_fold_5_baseline_valid_df)})


In [ ]:
# --- Step 23: Fold 5 TabNet baseline ---
dl_fold_5_tabnet_baseline_model = _fit_tabnet_baseline_runtime(
    dl_fold_5_baseline_train_df,
    list(dl_fold_5_bundle["feature_cols"]),
    random_seed=2105,
    valid_df=dl_fold_5_baseline_valid_df,
)
dl_fold_5_tabnet_baseline = run_dl_model_step(
    dl_fold_5_bundle,
    "dl_tabnet",
    "baseline",
    dl_fold_5_tabnet_baseline_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_5_tabnet_baseline)
print(dl_fold_5_tabnet_baseline["metric_row"])
finalize_dl_run(dl_fold_5_tabnet_baseline_model, dl_fold_5_tabnet_baseline)
cleanup_runtime_objects("dl_fold_5_tabnet_baseline_model", "dl_fold_5_tabnet_baseline")


In [ ]:
# --- Step 24: Fold 5 TabNet tuned ---
dl_fold_5_tabnet_tuned_model = fit_tabnet_tuned(
    pd.DataFrame(dl_fold_5_bundle["train_df"]),
    pd.DataFrame(dl_fold_5_bundle["inner_index"]),
    list(dl_fold_5_bundle["feature_cols"]),
    n_trials=int(DL_TRIALS_TABNET),
    random_seed=2205,
)
dl_fold_5_tabnet_tuned = run_dl_model_step(
    dl_fold_5_bundle,
    "dl_tabnet",
    "tuned",
    dl_fold_5_tabnet_tuned_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_5_tabnet_tuned)
print(dl_fold_5_tabnet_tuned["metric_row"])
finalize_dl_run(dl_fold_5_tabnet_tuned_model, dl_fold_5_tabnet_tuned)
cleanup_runtime_objects("dl_fold_5_tabnet_tuned_model", "dl_fold_5_tabnet_tuned")


In [ ]:
# --- Step 25: Fold 5 FT-Transformer baseline ---
dl_fold_5_ft_transformer_baseline_model = fit_ft_transformer_baseline(
    pd.DataFrame(dl_fold_5_bundle["train_df"]),
    list(dl_fold_5_bundle["feature_cols"]),
    random_seed=2305,
)
dl_fold_5_ft_transformer_baseline = run_dl_model_step(
    dl_fold_5_bundle,
    "dl_ft_transformer",
    "baseline",
    dl_fold_5_ft_transformer_baseline_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_5_ft_transformer_baseline)
print(dl_fold_5_ft_transformer_baseline["metric_row"])
finalize_dl_run(dl_fold_5_ft_transformer_baseline_model, dl_fold_5_ft_transformer_baseline)
cleanup_runtime_objects("dl_fold_5_ft_transformer_baseline_model", "dl_fold_5_ft_transformer_baseline")


In [ ]:
# --- Step 26: Fold 5 FT-Transformer tuned ---
dl_fold_5_ft_transformer_tuned_model = fit_ft_transformer_tuned(
    pd.DataFrame(dl_fold_5_bundle["train_df"]),
    pd.DataFrame(dl_fold_5_bundle["inner_index"]),
    list(dl_fold_5_bundle["feature_cols"]),
    n_trials=int(DL_TRIALS_FT),
    random_seed=2405,
)
dl_fold_5_ft_transformer_tuned = run_dl_model_step(
    dl_fold_5_bundle,
    "dl_ft_transformer",
    "tuned",
    dl_fold_5_ft_transformer_tuned_model,
    dl_artifact_dir,
)
append_dl_run_outputs(dl_prediction_frames, dl_metric_rows, dl_artifact_rows, dl_fold_5_ft_transformer_tuned)
print(dl_fold_5_ft_transformer_tuned["metric_row"])
finalize_dl_run(dl_fold_5_ft_transformer_tuned_model, dl_fold_5_ft_transformer_tuned)
cleanup_runtime_objects("dl_fold_5_ft_transformer_tuned_model", "dl_fold_5_ft_transformer_tuned")
cleanup_runtime_objects("dl_fold_5_bundle", "dl_fold_5_baseline_split_frames", "dl_fold_5_baseline_train_df", "dl_fold_5_baseline_valid_df")
print({"step": "dl_fold_completed", "outer_fold": 5})


In [ ]:
# --- Step 27: Write and verify DL outputs ---
write_dl_family_outputs(dl_result_dir, dl_prediction_frames, dl_metric_rows, dl_artifact_rows)
_assert_runtime_outputs_written(dl_result_dir)
print({"step": "dl_outputs_written", "files": expected_output_files(), "metric_rows": len(dl_metric_rows)})
